In [ ]:
import smolagents
from playpen.agents import ClemAgent, ClemObservation
from clemcore.backends import load_model
from clemcore.clemgame import env, episode_results_folder_callbacks
#from clemcore.backends import ModelRegistry, KeyRegistry

import os
import re
import json
import sys
import importlib
from clemcore.clemgame import ParseError, RuleViolationError
from pathlib import Path


This notebook was used to conduct a pilot experiment before testing the full Self-Refine pipeline.

The goal of **this pilot** was to measure the model's ability to analyze its own responses and give feedback on them.
The feedback step is crucial in Self-Refine, because it constructs the basis for the refinement step.
The model should get detailed information on what to refine specifically, not just vague instructions like
*improve this answer*. 

Bad feedback abilities would mean that the refinement has nothing specific to be based on, and hence, a poor
quality of refined answers can be expected.


For the testing, the *wordle* game was selected, as it demonstrated a significant amount of errors in the baseline,
so the models are still struggling with it, and because its Game Master and Game Scorer provide checks that can be
applied to every response on every turn.

## The Flow

The usual clembench `action`-`observation` exchange is retained. The context the Player gets is also
the same. However, the Player's responses do not get sent to the Game Master immediately. Instead, we first ask 
for the feedback (`def feedback`), and only then add the answer to the Player's memory and send it on.

To compare the feedback to reality, the Game Master's parsing functions are used --- they serve as gold
labels. The agreement is measured across three dimensions:

| | gold label from | 
|---|---|
| Format | `parse_response` | 
| Rules | `validate_guess` |
| Strategy | `turns_strategy` | 

There are small changes to the usage of the three functions which are described below in the docstrings. The
code itself remains untouched.

| gold_format | gold_rules | gold_strategy |
|-------------|------------|---------------|
| said_format | said_rules | said_strategy |

## Metrics

 - `balanced_accuracy` indicates whether a model can differentiate a faulty draft from a good one (measured per dimension)
 - `exact_agreement` as a proportion of faulty drafts where the dimensions flagged by the model match the gold labels exactly partly indicates 
 whether a model can tell **what exactly is wrong**. We restrict it to faulty drafts so that the "clean" responses do not artificially inflate
 the numbers. We are aware that this is a lower bound: getting two out of three dimensions right is also useful, but not counted here.

In [ ]:
CLEMBENCH_HOME = r""
GAME_NAME = "wordle"
os.environ["CLEMBENCH_HOME"] = CLEMBENCH_HOME
os.chdir("")    #path tp playpen is needed
os.environ["CUDA_VISIBLE_DEVICES"] = "5"

# import some wordle-specific functions from the game folder

WORDLE_DIR = os.path.join(CLEMBENCH_HOME, "wordle")
if WORDLE_DIR not in sys.path:
    sys.path.insert(0, WORDLE_DIR)
_wordle = importlib.import_module("master")

parse_response = _wordle.parse_response  
validate_guess = _wordle.validate_guess  
turns_strategy = _wordle.turns_strategy   


# official word list for the reponse parsing function
INSTANCES_PATH = os.path.join(CLEMBENCH_HOME, "wordle", "in", "instances.json")
with open(INSTANCES_PATH, encoding="utf-8") as f:
    WORDS = json.load(f)["experiments"][0]["lang_keywords"]
print(f"Import successful! {len(WORDS['official_words_list'])} words in the official list")


In [ ]:
from playpen.agents import ClemAgent, ClemObservation
from clemcore.backends import load_model

FEEDBACK_PROMPT = Path("").read_text(encoding="utf-8")

# helper: parse the model's feedbaxk
def parse_verdict(fb: str) -> dict:
    def get(label):
        m = re.search(rf"^{label}:\s*\"?(ok|problem)\"?", fb, re.I | re.M)
        return m.group(1).lower() if m else None
    return {"format":   get("FORMAT"),
            "rules":    get("RULES"),
            "strategy": get("STRATEGY"),
            "raw":      fb}

# helper: get the GM's evaluation for the draft response
def oracle_verdict(draft: str, last_feedback: str = None) -> dict:
    """What the Game Master's checks say about this draft, each run on its own.

    format     parse_response: starts with "explanation:", contains "guess:"
               (both exactly once)
    rules      validate_guess: one alphabetic word, five letters (and letters
                only), in the word list
    strategy   turns_strategy: the guess must not reuse letters the previous
                feedback marked red, weighed against greens and yellows kept
                (note: this is not an optimal strategy evaluation, and the
                oracle might score two answers equally even though one
                is better strategically under the circumstances)

    Two deliberate deviations: 
    
    1) usually, the Game Master never uses validate_guess 
    if the answer's format was wrong. However, we carry out both parse_response
    and validate_guess anyway, so that we can differentiate between format and
    rule failures.

    2) usually, the strategy is scored only after an episode is finished (it's
    originally a part of the Game Scorer). However, we invoke it once per turn
    during play.

    A dimension is None when its check could not run -- e.g., no guess to validate.
    Such cases when no gold label is accessible are excluded from the evaluation
    """
    verdict = {"format": "ok", "rules": "ok", "strategy": None,
               "reason": "", "strategy_score": None}
    reasons = []           # for the GM messages accompanying the errors

    # 1) the response format
    try:
        parse_response(None, draft, WORDS)    #note: this won't work for two-player games!
    except ParseError as error:
        verdict["format"] = "problem"
        reasons.append(error.reason)

    # 2) other rules
    keyword = re.escape(WORDS["guess_lang"])
    found = re.findall(rf"{keyword}([^\n]*)", draft, re.I)
    if not found:
        verdict["rules"] = None             # nothing to validate 
        verdict["reason"] = "; ".join(reasons)
        return verdict
    guess = found[0].strip().lower()
    try:
        validate_guess(guess, WORDS)
    except RuleViolationError as error:    
        verdict["rules"] = "problem"
        reasons.append(error.reason)

    # 3) the strategy (first turn is skipped, because there's no second guess
    # to compare with)
    if last_feedback:
        carrier = " ".join(f"{letter}<red>" for letter in guess)
        score = turns_strategy([last_feedback, carrier], is_aborted=False)[-1]
        verdict["strategy"] = "problem" if score <= 0 else "ok"
        verdict["strategy_score"] = score  
        if score <= 0:
            reasons.append(f"reuses letters already deemed wrong (strategy score {score:+})")
    else:
        verdict["strategy"] = None          # for the first guess in the episode

    verdict["reason"] = "; ".join(reasons)  
    return verdict


class MyAgenticGuesser(ClemAgent):
    """Produces a draft, asks itself for feedback on it, and records both the
    model's verdict and the Game Master's, so the two can be compared.
    No refinement happens: this is the feedback step on its own.
    """

    def __init__(self):
        super().__init__()
        self.model = load_model("llama-3.1-8b-instruct", gen_args=dict(temperature=0.0, max_tokens=600))
        self.verdicts = []
        self.episode = 1
        self.last_feedback = None

    def reset(self):
        # verdicts accumulate across episodes; the feedback must not
        super().reset()
        self.last_feedback = None
        self.episode += 1

    def feedback(self, feedback_prompt, initial_draft) -> str:
        """
        Using the same player model, generate feedback on the initial answer,
        evaluating it, to be further used for self-refinement
        """

        # Original Self-Refine equation: fbt =M(pfb ∥ x ∥ yt)
        feedback_msgs = self.history + [                             # x = self.history
            {"role": "assistant", "content": initial_draft},         # yt = draft
            {"role": "user",      "content": feedback_prompt}        # pfb
        ]

        first, second, feedback = self.model.generate_response(feedback_msgs)
        return feedback

    def act(self, last: ClemObservation) -> str:

        context = self.history[-1]["content"] if self.history else ""
        if any(m["role"] == "assistant" for m in self.history):
            seen = re.findall(rf"{re.escape(WORDS['guess_feedback_lang'])}([^\n]*)",
                              context, re.I)
            if seen:
                self.last_feedback = seen[-1].strip()

        # Original Self-Refine equation: y0 = M(pgen ∥ x)
        _, _, draft = self.model.generate_response(self.history)

        # get gold labels from the GM
        gold = oracle_verdict(draft, self.last_feedback)

        # original feedback step from Self-Refine
        feedback_result = self.feedback(FEEDBACK_PROMPT, draft)
        said = parse_verdict(feedback_result)

        turn = 1 + sum(1 for m in self.history if m["role"] == "assistant")
        self.observe(dict(role="assistant", content=draft))

        self.verdicts.append({"episode": self.episode,
                              "experiment": self.experiment,
                              "game_id": self.game_id,
                              "turn": turn,
                              "draft": draft,
                              "gold_format": gold["format"],
                              "gold_rules": gold["rules"],
                              "gold_strategy": gold["strategy"],
                              "gold_reason": gold["reason"],
                              "gold_strategy_score": gold["strategy_score"],
                              "said_format": said["format"],
                              "said_rules": said["rules"],
                              "said_strategy": said["strategy"],
                              "said_raw": said["raw"]})

        return draft


guesser = MyAgenticGuesser()


In [ ]:
from clemcore.clemgame import env, episode_results_folder_callbacks

# Create callbacks to record the interactions in a folder; here we name the folder after the models the agent uses
callbacks = episode_results_folder_callbacks(run_dir="llama-3.1-8b-instruct", result_dir_path="playpen-records", player_model_infos="MyAgenticGuesser")

game_env = env(
    GAME_NAME,
    single_pass=True,
    callbacks=callbacks
)
agent_mapping = {"player_0": guesser}


In [ ]:
# Now we can iterate through the agent turns, one step at a time

episodes = 0
while episodes < 30:
    game_env.reset()
    guesser.experiment = game_env.unwrapped.experiment["name"]
    guesser.game_id = game_env.unwrapped.game_instance.get("game_id")
    for agent_id in game_env.agent_iter():
        context, reward, termination, truncation, info = game_env.last()
        response = None if (termination or truncation) else agent_mapping[agent_id](context)
        game_env.step(response)
    guesser.reset()
    episodes += 1
    print(f"episode {episodes}: verdicts collected: {len(guesser.verdicts)}")


In [ ]:
# Compare what the Game Master's checks said with what the model said about its own draft

import pandas as pd
from sklearn.metrics import balanced_accuracy_score

DIMENSIONS = ("format", "rules", "strategy")

rows = pd.DataFrame(guesser.verdicts)
rows.to_csv(f"feedback_pilot_verdicts_{guesser.model}.csv", index=False)

print(f"{len(rows)} drafts collected\n")

for dimension in DIMENSIONS:
    # gold label-cells where a dimension is None are exluded from the evaluation
    scorable = rows[rows[f"gold_{dimension}"].notna()]
    untested = len(rows) - len(scorable)
    violations = scorable[scorable[f"gold_{dimension}"] == "problem"]
    clean = scorable[scorable[f"gold_{dimension}"] == "ok"]
    if len(violations) == 0:
        print(f"{dimension:9s} no violations yet -- run more episodes"
              + (f" ({untested} untested)" if untested else ""))
        continue
    hit = (violations[f"said_{dimension}"] == "problem").mean()
    line = (f"{dimension:9s} n={len(violations):3d} problems, {len(clean):3d} clean"
            + (f", {untested:3d} untested" if untested else "")
            + f" | caught {hit:5.1%}")
    if len(clean) == 0:
        print(line + " | no clean items yet")
    else:
        y_true = (scorable[f"gold_{dimension}"] == "problem").astype(int)
        y_pred = (scorable[f"said_{dimension}"] == "problem").astype(int)
        false_alarm = (clean[f"said_{dimension}"] == "problem").mean()
        acc = balanced_accuracy_score(y_true, y_pred)
        print(line + f" | false alarms {false_alarm:5.1%} | Balanced accuracy {acc:.3f}")
print()


def flagged_set(row, prefix):
    names = [d for d in DIMENSIONS if row[f"{prefix}_{d}"] == "problem"]
    return ", ".join(names) or "(nothing)"


# Exact agreement: does the model flag the same set of dimensions as the gold labels
complete = rows[[all(pd.notna(r[f"gold_{d}"]) for d in DIMENSIONS)
                 for _, r in rows.iterrows()]]
if len(complete):
    gold_sets = complete.apply(lambda r: flagged_set(r, "gold"), axis=1)
    said_sets_known = complete.apply(lambda r: flagged_set(r, "said"), axis=1)
    print(f"exact agreement on which dimensions are at fault: "
          f"{(gold_sets == said_sets_known).mean():.1%} of {len(complete)} drafts\n")
    print("oracle (rows) against model (columns):")
    print(pd.crosstab(gold_sets, said_sets_known).to_string())
    print()

said_sets = rows.apply(lambda r: flagged_set(r, "said"), axis=1)
print("distinct verdicts the model produced:")
print(said_sets.value_counts().to_string())
